# Week 4: Sentinel-3 Altimetry Classification

This notebook classifies Sentinel-3 SAR altimetry echoes into sea ice and leads using a two-component Gaussian Mixture Model. The classification is compared with the ESA surface-type classification, and the mean echo shape and standard deviation are calculated for both classes.

The analysis is based on the supplied Week 4 unsupervised-learning notebook and Sentinel-3 course data.


## Data

The analysis uses the supplied Sentinel-3 Level-2 LAND ice product. The `surf_type_class_20_ku` variable defines 1 as sea ice and 2 as lead. The classification features are backscatter (`sig0_water_20_ku`), waveform peakiness and stack standard deviation derived from `rip_20_ku`.


In [ ]:
!pip install netCDF4 -q

from google.colab import drive
drive.mount('/content/drive')

from netCDF4 import Dataset
import glob
import zipfile
import warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import confusion_matrix


## Functions


In [ ]:
def peakiness(waves):
    def by_row(wave):
        maximum = np.nanmax(wave)
        if not np.isfinite(maximum) or maximum <= 0:
            return np.nan

        maximum_bin = np.where(wave == maximum)[0][0]
        segment = wave[maximum_bin-50:maximum_bin+78]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            noise_floor = np.nanmean(segment[10:20])

        above = np.where(segment > noise_floor)[0]
        if above.size == 0:
            return np.nan

        return np.nanmax(segment[above]) / np.nanmean(segment[above])

    return np.apply_along_axis(by_row, 1, np.asarray(waves))


def unpack_gpod(variable):
    time_1hz = SAR_data.variables['time_01'][:]
    time_20hz = SAR_data.variables['time_20_ku'][:]
    time_20hz_c = SAR_data.variables['time_20_c'][:]

    out = SAR_data.variables[variable][:].astype(float)
    out = np.ma.filled(out, np.nan)

    if len(out) == len(time_1hz):
        out = interp1d(time_1hz, out, fill_value="extrapolate")(time_20hz)

    if len(out) == len(time_20hz_c):
        out = interp1d(time_20hz_c, out, fill_value="extrapolate")(time_20hz)

    return out


def calculate_SSD(rip):
    def gaussian(x, a, x0, sigma):
        return a * np.exp(-(x - x0) ** 2 / (2 * sigma ** 2))

    ssd = np.full(rip.shape[0], np.nan)
    x = np.arange(rip.shape[1])

    for i in range(rip.shape[0]):
        y = np.array(rip[i], copy=True)
        y[np.isnan(y)] = 0

        total = np.sum(y)
        if total == 0:
            continue

        mean_est = np.sum(x * y) / total
        sigma_est = np.sqrt(np.sum(y * (x - mean_est) ** 2) / total)

        try:
            popt, _ = curve_fit(
                gaussian,
                x,
                y,
                p0=[np.max(y), mean_est, sigma_est],
                maxfev=10000
            )
            ssd[i] = abs(popt[2])
        except (RuntimeError, TypeError, ValueError):
            pass

    return ssd


## Load and preprocess the Sentinel-3 data


In [ ]:
path = '/content/drive/MyDrive/GEOL0069_Week4/'

s3_zip = glob.glob(path + 'S3A_SR_2_LAN_SI_*.zip')[0]

with zipfile.ZipFile(s3_zip, 'r') as z:
    nc_name = [name for name in z.namelist() if name.endswith('enhanced_measurement.nc')][0]
    z.extract(nc_name, '/content/week4_s3')

SAR_data = Dataset('/content/week4_s3/' + nc_name)

SAR_lat = unpack_gpod('lat_20_ku')
SAR_lon = unpack_gpod('lon_20_ku')
waves = unpack_gpod('waveform_20_ku')
sig_0 = unpack_gpod('sig0_water_20_ku')
RIP = unpack_gpod('rip_20_ku')
flag = unpack_gpod('surf_type_class_20_ku')

valid = np.where(SAR_lat >= -99999)
SAR_lat = SAR_lat[valid]
SAR_lon = SAR_lon[valid]
waves = waves[valid]
sig_0 = sig_0[valid]
RIP = RIP[valid]
flag = flag[valid]

PP = peakiness(waves)
SSD = calculate_SSD(RIP)

data = np.column_stack((np.asarray(sig_0), np.asarray(PP), np.asarray(SSD)))
data_normalized = StandardScaler().fit_transform(data)

nan_rows = np.isnan(data_normalized).any(axis=1)
data_cleaned = data_normalized[~nan_rows]
flag_cleaned = flag[~nan_rows]
waves_cleaned_all = waves[~nan_rows]

class_mask = (flag_cleaned == 1) | (flag_cleaned == 2)
data_class = data_cleaned[class_mask]
flag_class = flag_cleaned[class_mask]
waves_cleaned = waves_cleaned_all[class_mask]

print('Waveforms:', waves.shape)
print('NaN values removed:', int(np.isnan(data_normalized).sum()))
print('Valid ESA sea-ice/lead echoes:', data_class.shape[0])


Waveforms: (13101, 256)
NaN values removed: 1283
Valid ESA sea-ice/lead echoes: 12195


## Two-class Gaussian Mixture Model

Only observations labelled by ESA as sea ice or lead are used for the direct comparison. The three input features are standardized before fitting the model.


In [ ]:
gmm = GaussianMixture(n_components=2, random_state=0)
gmm.fit(data_class)
clusters_gmm = gmm.predict(data_class)

unique, counts = np.unique(clusters_gmm, return_counts=True)
class_counts = dict(zip(unique, counts))

print(class_counts)


{0: 8880, 1: 3315}


## Comparison with ESA classification

The GMM cluster numbering is arbitrary. Comparison with the ESA surface-type flags shows that cluster 0 corresponds to sea ice and cluster 1 corresponds to leads.


In [ ]:
esa_binary = np.where(flag_class == 1, 0, 1)
cm = confusion_matrix(esa_binary, clusters_gmm)
agreement = np.trace(cm) / np.sum(cm) * 100

print(cm)
print(f'Agreement with ESA classification: {agreement:.2f}%')

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.imshow(cm)
ax.set_xticks([0, 1], labels=['Sea ice', 'Lead'])
ax.set_yticks([0, 1], labels=['Sea ice', 'Lead'])
ax.set_xlabel('GMM classification')
ax.set_ylabel('ESA classification')
ax.set_title('GMM versus ESA classification')

for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]}', ha='center', va='center')

plt.tight_layout()
plt.savefig(path + 'Week4_confusion_matrix.png', dpi=300)
plt.show()


[[8856   22]
 [  24 3293]]
Agreement with ESA classification: 99.62%


The confusion matrix contains 12,149 agreements and 46 disagreements across 12,195 classified echoes, giving 99.62% agreement with the ESA classification.


## Mean echo shape and standard deviation


In [ ]:
sea_ice_waves = waves_cleaned[clusters_gmm == 0]
lead_waves = waves_cleaned[clusters_gmm == 1]

sea_ice_mean = np.mean(sea_ice_waves, axis=0)
sea_ice_std = np.std(sea_ice_waves, axis=0)
lead_mean = np.mean(lead_waves, axis=0)
lead_std = np.std(lead_waves, axis=0)

x = np.arange(waves_cleaned.shape[1])

sea_lower = np.maximum(sea_ice_mean - sea_ice_std, 0)
sea_upper = sea_ice_mean + sea_ice_std
lead_lower = np.maximum(lead_mean - lead_std, 0)
lead_upper = lead_mean + lead_std

plt.figure(figsize=(10, 6))
plt.plot(x, sea_ice_mean, label='Sea ice mean')
plt.fill_between(x, sea_lower, sea_upper, alpha=0.25)
plt.plot(x, lead_mean, label='Lead mean')
plt.fill_between(x, lead_lower, lead_upper, alpha=0.25)

plt.xlabel('Waveform bin')
plt.ylabel('Echo power')
plt.title('Average Sentinel-3 Echo Shapes: Sea Ice and Leads')
plt.legend()
plt.tight_layout()
plt.savefig(path + 'Week4_average_echo_shapes.png', dpi=300)
plt.show()


Lead echoes have a sharper and higher-amplitude mean peak, while sea-ice echoes have a broader and lower-amplitude mean response. The wider standard-deviation envelope for leads indicates greater variation within the lead class.


## Conclusion

The two-component GMM separated the Sentinel-3 echoes into sea ice and leads with close agreement to the ESA surface-type classification. The confusion matrix gives 99.62% agreement, and the mean waveform comparison shows a clear difference in echo shape between the two surface types.
